# Quantitative XAI Evaluation for AlphaDent YOLO26 (v5)

v5 architecture change: **one** differentiable train-mode forward+backward per image; all 7 CAMs
derived from the cached (activation, gradient) pair — 7x fewer graphs, graph freed immediately
(success or failure). Dict-aware output flattening (ultralytics train-mode returns a dict).
Per-config model-load guard. This eliminates the v4 OOM cascade.

Deletion/insertion faithfulness + inter-method agreement per config; combined CSVs, figures,
LaTeX rows, one zip.

In [ ]:
# Clone repo if not running inside it already
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'  # reduce CUDA fragmentation
if not os.path.exists('/kaggle/working/AlphaDentYolov26'):
    print("Cloning AlphaDentYolov26 repository...")
    !git clone https://github.com/ShMazumder/AlphaDentYolov26 /kaggle/working/AlphaDentYolov26/
else:
    print("Repository already exists.")

# Change directory to the repository root
os.chdir('/kaggle/working/AlphaDentYolov26/')
print(f"Current working directory: {os.getcwd()}")

# Install dependencies
print("Installing dependencies...")
!pip install -q -r requirements.txt
!pip install -q matplotlib opencv-python scipy pandas


In [ ]:
import zipfile
import shutil
import os

input_dir = "/kaggle/input/competitions/alpha-dent/AlphaDent"
working_dir = "/kaggle/working/AlphaDentYolov26/AlphaDent"
zip_path = "/kaggle/working/dataset.zip"

# Check if Kaggle input contains the dataset
if os.path.exists(os.path.join(input_dir, 'images')) and os.listdir(os.path.join(input_dir, 'images')):
    print("Dataset found in Kaggle input. Copying dataset...")
    shutil.copytree(input_dir, working_dir, dirs_exist_ok=True)
    print("Dataset copied successfully. Using input dataset.")
else:
    # Fallback to downloading from Zenodo
    images_dir = os.path.join(working_dir, 'images')
    if not os.path.exists(images_dir) or not os.listdir(images_dir):
        print("Downloading AlphaDent dataset from Zenodo...")
        !wget -q --show-progress "https://zenodo.org/records/16582489/files/AlphaDent.zip?download=1" -O {zip_path}

        print("Extracting dataset...")
        os.makedirs(working_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(working_dir)

        # Remove zip to save disk space
        if os.path.exists(zip_path):
            os.remove(zip_path)

        # Check if nested folder exists after extraction and move files up if needed
        nested_dir = os.path.join(working_dir, 'AlphaDent')
        if os.path.exists(nested_dir) and os.path.isdir(nested_dir):
            print("Adjusting dataset directory structure...")
            for item in os.listdir(nested_dir):
                shutil.move(os.path.join(nested_dir, item), working_dir)
            os.rmdir(nested_dir)

        print("Dataset prepared successfully in working directory.")
    else:
        print("Dataset already exists in working directory.")

# NOTE: no 4-class label conversion here — this eval needs the ORIGINAL 9-class labels.
# The runner remaps ground truth (cls>=3 -> 3) on the fly when evaluating nc==4 models,
# so a single unconverted label set serves both model families. Converting in place
# would silently break TP/FP scoring for all 9-class configs.


In [ ]:
import os, re, glob, torch

# ---- Config: discover ALL trained weights ----
CANDIDATE_GLOBS = [
    'runs/segment/*/train/weights/best.pt',
    'runs_4classes/segment/*/train/weights/best.pt',
    '/kaggle/working/AlphaDentYolov26/runs*/segment/*/train/weights/best.pt',
    '/kaggle/input/*/**/best.pt',
]
found = []
for g in CANDIDATE_GLOBS:
    found += glob.glob(g, recursive=True)
found = sorted(set(found))

def cfg_name(path):
    m = re.search(r'yolo_seg_(yolo26[nsmlx])_proj_(\d+)', path)
    if m:
        tag = '4class_' if ('4class' in path.lower()) else ''
        return f"{tag}{m.group(1)}_{m.group(2)}", int(m.group(2))
    return os.path.basename(os.path.dirname(os.path.dirname(path))), 640

CONFIGS = []   # (name, weights_path, imgsz)
seen = set()
for p in found:
    name, sz = cfg_name(p)
    if name not in seen:
        seen.add(name)
        CONFIGS.append((name, p, sz))
# Pipeline filter: 9-class only
CONFIGS = [c for c in CONFIGS if ('4class' in c[0].lower() or '4class' in c[1].lower()) == False]
assert CONFIGS, 'No 9-class best.pt found (4-class weight paths must contain \'4class\').'

EXPECTED_NC = 9  # checkpoint-level family check (path naming can lie)
DATA_DIR = './AlphaDent'
N_IMAGES = 30
N_STEPS  = 20
OUT_DIR  = 'xai_eval'
DEVICE   = 0 if torch.cuda.is_available() else 'cpu'
METHODS  = ['EigenCAM', 'GradCAM', 'GradCAM++', 'XGradCAM', 'HiResCAM', 'LayerCAM', 'EigenGradCAM']

os.makedirs(OUT_DIR, exist_ok=True)
print(f'{len(CONFIGS)} configs discovered:')
for n, p, sz in CONFIGS:
    print(f'  {n:24s} imgsz={sz}  {p}')


In [ ]:
import os, cv2, numpy as np, torch

def letterbox(img, size, pad_val=114):
    h, w = img.shape[:2]
    r = size / max(h, w)
    nh, nw = int(h * r), int(w * r)
    rs = cv2.resize(img, (nw, nh))
    canvas = np.full((size, size, 3), pad_val, dtype=np.uint8)
    top, left = (size - nh) // 2, (size - nw) // 2
    canvas[top:top + nh, left:left + nw] = rs
    return canvas, r, (top, left)

def to_tensor(img_bgr, device):
    x = img_bgr.astype(np.float32) / 255.0
    return torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).to(device)

def unwrap_tensor(out):
    """Descend nested list/tuple output until first Tensor."""
    while isinstance(out, (list, tuple)):
        out = out[0]
    return out

def parse_yolo_preds(t, nc):
    """Normalize YOLO output. Returns (kind, mat):
       kind='raw': mat (C,N) channel-first grids, channels = [box4, cls nc, coeff...]
       kind='e2e': mat (N,K) decoded rows = [x1,y1,x2,y2,conf,cls,(coeff...)]"""
    if t.ndim == 3:
        t = t[0]
    if t.ndim != 2:
        raise ValueError(f'unexpected pred shape {tuple(t.shape)}')
    r, c = t.shape
    C_raw = 4 + nc + 32          # channel-first grid width (box+cls+mask coeffs)
    if r == C_raw or r == 4 + nc:
        return 'raw', t
    if c == C_raw or c == 4 + nc:
        return 'raw', t.T
    if c in (6, 38):              # decoded rows: 6 cols, or 6+32 with mask coeffs
        return 'e2e', t
    if r in (6, 38):
        return 'e2e', t.T
    # fallback heuristics
    if r < c and r >= 4 + nc:
        return 'raw', t
    if c < r and c >= 4 + nc:
        return 'raw', t.T
    raise ValueError(f'cannot parse pred shape {tuple(t.shape)} with nc={nc}')

def pred_class_and_score(t, nc, class_idx=None):
    """Differentiable: returns (class_idx, scalar score tensor for that class)."""
    kind, mat = parse_yolo_preds(t, nc)
    if kind == 'raw':
        if class_idx is None:
            class_idx = int(mat[4:4 + nc, :].sum(dim=-1).argmax())
        return class_idx, mat[4 + class_idx, :].max()
    det = mat                      # rows: [x1,y1,x2,y2,conf,cls,...]
    if class_idx is None:
        class_idx = int(det[det[:, 4].argmax(), 5])
    m = det[:, 5].long() == class_idx
    score = det[m, 4].max() if m.any() else det[:, 4].sum() * 0.0
    return class_idx, score

TRAPZ = getattr(np, 'trapezoid', None) or np.trapz

def collect_cls_logit_maps(out, nc, acc=None):
    """Recursively collect 4-D head tensors from a TRAIN-mode forward whose channel
    count can contain nc class logits at the tail (excludes 32-ch mask-coeff maps)."""
    if acc is None:
        acc = []
    if torch.is_tensor(out):
        if out.ndim == 4 and out.shape[1] > max(nc, 32):
            acc.append(out)
    elif isinstance(out, dict):
        for o in out.values():
            collect_cls_logit_maps(o, nc, acc)
    elif isinstance(out, (list, tuple)):
        for o in out:
            collect_cls_logit_maps(o, nc, acc)
    return acc

@torch.no_grad()
def class_score(yolo, img_bgr, class_idx, device, nc):
    out = unwrap_tensor(yolo.model(to_tensor(img_bgr, device)))
    _, s = pred_class_and_score(out, nc, class_idx)
    return float(s.cpu())

def poly_txt_to_boxes(txt_path, img_w, img_h):
    boxes = []
    if not os.path.exists(txt_path):
        return boxes
    for line in open(txt_path):
        p = line.split()
        if len(p) < 7:
            continue
        cls = int(p[0])
        xs = np.array(p[1::2], dtype=float) * img_w
        ys = np.array(p[2::2], dtype=float) * img_h
        boxes.append((cls, xs.min(), ys.min(), xs.max(), ys.max()))
    return boxes

def box_iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def _flatten_tensors(out, acc=None):
    if acc is None:
        acc = []
    if torch.is_tensor(out):
        acc.append(out)
    elif isinstance(out, dict):
        for o in out.values():
            _flatten_tensors(o, acc)
    elif isinstance(out, (list, tuple)):
        for o in out:
            _flatten_tensors(o, acc)
    return acc

def cam_from_act_grad(method, act, grad):
    """All 7 CAMs from one cached (act, grad) pair. act/grad: (C,H,W) numpy."""
    m = method.lower()
    if m == 'eigencam':
        C, H, W = act.shape
        M = act.reshape(C, -1).T; M = M - M.mean(0)
        U, _, _ = np.linalg.svd(M, full_matrices=False)
        p = U[:, 0].reshape(H, W)
    elif m == 'gradcam':
        w = grad.mean(axis=(1, 2))
        p = np.maximum((w[:, None, None] * act).sum(0), 0)
    elif m == 'gradcam++':
        g2, g3 = grad ** 2, grad ** 3
        sa = act.sum(axis=(1, 2))
        alpha = g2 / (2 * g2 + sa[:, None, None] * g3 + 1e-8)
        w = (alpha * np.maximum(grad, 0)).sum(axis=(1, 2))
        p = np.maximum((w[:, None, None] * act).sum(0), 0)
    elif m == 'xgradcam':
        sa = act.sum(axis=(1, 2))
        w = (grad * act).sum(axis=(1, 2)) / (sa + 1e-8)
        p = np.maximum((w[:, None, None] * act).sum(0), 0)
    elif m == 'hirescam':
        p = np.maximum((act * grad).sum(0), 0)
    elif m == 'layercam':
        p = np.maximum((act * np.maximum(grad, 0)).sum(0), 0)
    elif m == 'eigengradcam':
        AG = act * np.maximum(grad, 0)
        C, H, W = AG.shape
        M = AG.reshape(C, -1).T; M = M - M.mean(0)
        U, _, _ = np.linalg.svd(M, full_matrices=False)
        p = U[:, 0].reshape(H, W)
    else:
        raise ValueError(method)
    return (p - p.min()) / (p.max() - p.min() + 1e-8)

def compute_act_grad(model, target_layer, input_tensor, nc):
    """ONE eval-mode differentiable forward+backward -> (class_idx, act, grad) numpy.
    Target = the top detection's class confidence (decoded output via pred_class_and_score).
    This is the CORRECT class channel and is downstream of the whole neck, so the gradient
    is guaranteed to reach the hooked layer. Replaces the earlier train-mode raw-map
    indexing, which (a) mis-indexed mask-coefficient channels -> HiRes==Layer / Grad==XGrad
    collapse, and (b) sometimes backpropped through a scale that bypassed the hooked layer
    -> KeyError('grad'). Graph freed before returning."""
    store = {}
    fh = target_layer.register_forward_hook(
        lambda mod, i, o: store.__setitem__('act', o[0] if isinstance(o, (list, tuple)) else o))
    bh = target_layer.register_full_backward_hook(
        lambda mod, gi, go: store.__setitem__('grad', go[0] if isinstance(go, (list, tuple)) else go))
    net = model.model  # keep EVAL mode (set by caller); do NOT switch to train
    try:
        x = input_tensor.clone().requires_grad_(True)
        out = unwrap_tensor(net(x))
        class_idx, score = pred_class_and_score(out, nc, None)
        net.zero_grad(set_to_none=True)
        score.backward()
        if 'grad' not in store:
            raise RuntimeError('backward hook on target layer did not fire '
                               '(score path bypasses the hooked layer)')
        act  = store['act'].detach().float().cpu().numpy()[0]
        grad = store['grad'].detach().float().cpu().numpy()[0]
    finally:
        store.clear()
        net.zero_grad(set_to_none=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        fh.remove(); bh.remove()
    return class_idx, act, grad


In [ ]:
# ---- Run EVERYTHING: faithfulness + agreement for every discovered config ----
import pandas as pd
from itertools import combinations
from scipy.stats import spearmanr, mannwhitneyu
from ultralytics import YOLO

# Defensive cleanup: drop stale globals from earlier cell runs in this kernel
import gc as _gc
for _g in ['model', 'heat_store', '_raw', '_t', 'fdf', 'adf']:
    if _g in dir():
        del globals()[_g]
_gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_b, total_b = torch.cuda.mem_get_info()
    print(f'GPU free before start: {free_b/1e9:.1f} / {total_b/1e9:.1f} GB')
    assert free_b > 4e9, 'GPU mostly occupied — RESTART THE KERNEL and Run All.'

val_images = sorted(glob.glob(f'{DATA_DIR}/images/valid/*.jpg') + glob.glob(f'{DATA_DIR}/images/valid/*.png'))[:N_IMAGES]
print(f'{len(val_images)} eval images')

def run_faithfulness(model, nc, target_layer, imgsz, out_sub):
    rows, heat_store = [], {}
    for ip in val_images:
        img = cv2.imread(ip)
        img_lb, _, _ = letterbox(img, imgsz)
        # ONE differentiable pass per image -> act/grad; all 7 CAMs derived from it
        try:
            cls, act, grad = compute_act_grad(model, target_layer, to_tensor(img_lb, DEVICE), nc)
        except Exception as e:
            print('  skip', os.path.basename(ip), repr(e))
            import gc as _gc; _gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            continue
        base = class_score(model, img_lb, cls, DEVICE, nc) + 1e-8
        blur = cv2.GaussianBlur(img_lb, (51, 51), 0)
        h, w = img_lb.shape[:2]
        flat_img, flat_blur = img_lb.reshape(-1, 3), blur.reshape(-1, 3)
        ins0 = class_score(model, blur, cls, DEVICE, nc) / base
        for m in METHODS:
            try:
                heat = cam_from_act_grad(m, act, grad)
                hm = cv2.resize(heat, (w, h)).flatten()
                order = np.argsort(-hm)
                dele, ins = [1.0], [ins0]
                for k in range(1, N_STEPS + 1):
                    idx = order[: int(len(order) * k / N_STEPS)]
                    d = flat_img.copy(); d[idx] = 114
                    i = flat_blur.copy(); i[idx] = flat_img[idx]
                    dele.append(class_score(model, d.reshape(h, w, 3), cls, DEVICE, nc) / base)
                    ins.append(class_score(model, i.reshape(h, w, 3), cls, DEVICE, nc) / base)
                rows.append({'image': os.path.basename(ip), 'method': m, 'class': cls,
                             'deletion_auc': float(TRAPZ(dele, dx=1/N_STEPS)),
                             'insertion_auc': float(TRAPZ(ins, dx=1/N_STEPS))})
                heat_store[(ip, m)] = heat
            except Exception as e:
                print('  fail', m, os.path.basename(ip), repr(e))
        del act, grad
        import gc as _gc; _gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if rows:
            pd.DataFrame(rows).to_csv(f'{out_sub}/faithfulness_raw.csv', index=False)
    return pd.DataFrame(rows), heat_store

def run_agreement(model, nc, imgsz, heat_store, out_sub):
    rows = []
    for ip in val_images:
        hs = [cv2.resize(heat_store[(ip, m)], (64, 64)).flatten() for m in METHODS if (ip, m) in heat_store]
        if len(hs) < 2:
            continue
        ag = float(np.nanmean([spearmanr(a, b).correlation for a, b in combinations(hs, 2)]))
        img = cv2.imread(ip); H, W = img.shape[:2]
        r = model.predict(ip, imgsz=imgsz, conf=0.25, verbose=False, device=DEVICE)[0]
        correct, conf = 0, 0.0
        if len(r.boxes):
            b = r.boxes[r.boxes.conf.argmax()]
            conf = float(b.conf)
            pcls = int(b.cls); pbox = b.xyxy[0].tolist()
            gts = poly_txt_to_boxes(f"{DATA_DIR}/labels/valid/{os.path.splitext(os.path.basename(ip))[0]}.txt", W, H)
            if nc == 4:  # model trained on merged labels; remap 9-class GT
                gts = [(min(g[0], 3), *g[1:]) for g in gts]
            correct = int(any(g[0] == pcls and box_iou(pbox, g[1:]) >= 0.5 for g in gts))
        rows.append({'image': os.path.basename(ip), 'agreement': ag, 'top_conf': conf, 'correct': correct})
    df = pd.DataFrame(rows)
    df.to_csv(f'{out_sub}/agreement.csv', index=False)
    return df

all_faith, all_agree = [], []
for name, wpath, imgsz in CONFIGS:
    print(f'\n===== {name} (imgsz={imgsz}) =====')
    out_sub = f'{OUT_DIR}/{name}'; os.makedirs(out_sub, exist_ok=True)
    import gc as _gc; _gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        model = YOLO(wpath); model.model.to(DEVICE).eval()
    except Exception as e:
        print('  cannot load model (skipping config):', repr(e)); continue
    nc = getattr(model.model, 'nc', None) or len(model.names)
    if 'EXPECTED_NC' in globals() and nc != EXPECTED_NC:
        print(f'  SKIP: checkpoint has nc={nc}, this notebook evaluates nc={EXPECTED_NC} models '
              f'(wrong-family weights at {wpath})')
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue
    target_layer = model.model.model[22]
    # diagnostic once per config
    _lb, _, _ = letterbox(cv2.imread(val_images[0]), imgsz)
    with torch.no_grad():
        _t = unwrap_tensor(model.model(to_tensor(_lb, DEVICE)))
    print(f'  nc={nc} raw shape={tuple(_t.shape)} parsed={parse_yolo_preds(_t, nc)[0]}')
    fdf, heat_store = run_faithfulness(model, nc, target_layer, imgsz, out_sub)
    if fdf.empty:
        print('  no faithfulness rows, skipping agreement'); continue
    fdf['config'] = name; all_faith.append(fdf)
    adf = run_agreement(model, nc, imgsz, heat_store, out_sub)
    if adf.empty:
        print('  no agreement rows (need >=2 methods per image)'); continue
    adf['config'] = name; all_agree.append(adf)
    tp, fp = adf[adf.correct == 1].agreement, adf[adf.correct == 0].agreement
    msg = f'  agreement: TP={tp.mean():.3f}(n={len(tp)}) FP={fp.mean():.3f}(n={len(fp)})'
    if len(tp) > 2 and len(fp) > 2:
        u, pv = mannwhitneyu(tp, fp, alternative='greater')
        msg += f'  MWU p={pv:.4f}'
    print(msg)
    del model, heat_store
    if torch.cuda.is_available(): torch.cuda.empty_cache()

assert all_faith, 'Nothing produced.'
faith = pd.concat(all_faith)
agree = pd.concat(all_agree) if all_agree else pd.DataFrame(columns=['image','agreement','top_conf','correct','config'])
faith.to_csv(f'{OUT_DIR}/faithfulness_all.csv', index=False)
agree.to_csv(f'{OUT_DIR}/agreement_all.csv', index=False)
print('\nCombined summary (mean deletion/insertion AUC per config x method):')
print(faith.groupby(['config','method'])[['deletion_auc','insertion_auc']].mean().round(4))


In [ ]:
# ---- Aggregate outputs: figures, LaTeX rows, stats, zip ----
import pandas as pd, matplotlib.pyplot as plt, shutil
from scipy.stats import spearmanr, mannwhitneyu

faith = pd.read_csv(f'{OUT_DIR}/faithfulness_all.csv')
agree = pd.read_csv(f'{OUT_DIR}/agreement_all.csv')

# Method-level summary pooled across configs
pooled = faith.groupby('method')[['deletion_auc','insertion_auc']].agg(['mean','std']).round(4)
pooled.to_csv(f'{OUT_DIR}/faithfulness_pooled.csv')
print(pooled)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
faith.boxplot(column='deletion_auc', by='method', ax=axes[0], rot=45)
axes[0].set_title('Deletion AUC (lower = faithful)'); axes[0].set_xlabel('')
faith.boxplot(column='insertion_auc', by='method', ax=axes[1], rot=45)
axes[1].set_title('Insertion AUC (higher = faithful)'); axes[1].set_xlabel('')
plt.suptitle(''); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_faithfulness.png', dpi=200); plt.show()

plt.figure(figsize=(5.5, 4.5))
for c, lbl, col in [(1, 'correct (TP)', 'tab:green'), (0, 'incorrect/miss', 'tab:red')]:
    sub = agree[agree.correct == c]
    plt.scatter(sub.agreement, sub.top_conf, label=lbl, c=col, alpha=0.6, s=18)
plt.xlabel('inter-method agreement (mean pairwise Spearman)')
plt.ylabel('top prediction confidence'); plt.legend(); plt.tight_layout()
plt.savefig(f'{OUT_DIR}/fig_agreement.png', dpi=200); plt.show()

tp, fp = agree[agree.correct == 1].agreement, agree[agree.correct == 0].agreement
print(f'\nPooled agreement: TP={tp.mean():.3f}(n={len(tp)}) FP={fp.mean():.3f}(n={len(fp)})')
if len(tp) > 2 and len(fp) > 2:
    u, pv = mannwhitneyu(tp, fp, alternative='greater')
    print(f'Mann-Whitney U (TP > FP): U={u:.0f}, p={pv:.5f}')
print(f'Spearman(agreement, conf) = {spearmanr(agree.agreement, agree.top_conf).correlation:.3f}')

print('\n% --- LaTeX rows: pooled faithfulness (method & del AUC & ins AUC) ---')
for m in pooled.index:
    print(f"{m} & {pooled.loc[m,('deletion_auc','mean')]:.3f} $\\pm$ {pooled.loc[m,('deletion_auc','std')]:.3f} & "
          f"{pooled.loc[m,('insertion_auc','mean')]:.3f} $\\pm$ {pooled.loc[m,('insertion_auc','std')]:.3f} \\\\")

dst = '/kaggle/working/xai_eval_results' if os.path.exists('/kaggle/working') else 'xai_eval_results'
shutil.make_archive(dst, 'zip', OUT_DIR)
print('\nzipped ->', dst + '.zip')


In [ ]:
# ------- F1/F3 post-run diagnostics & figure (added by review fix) -------


In [ ]:
# === F1 verification: are the 7 CAMs actually distinct now? ===
# Run AFTER the main config loop (uses last-loaded `model`, `target_layer`, `nc`, `imgsz`).
import numpy as np, cv2
METHODS = ['EigenCAM','GradCAM','GradCAM++','XGradCAM','HiResCAM','LayerCAM','EigenGradCAM']
_img = cv2.imread(val_images[0]); _lb,_,_ = letterbox(_img, imgsz)
_cls,_act,_grad = compute_act_grad(model, target_layer, to_tensor(_lb, DEVICE), nc)
_cams = {m: cam_from_act_grad(m,_act,_grad).ravel() for m in METHODS}
print('pairwise cosine similarity (1.000 == identical map):')
for i in range(len(METHODS)):
    for j in range(i+1,len(METHODS)):
        a,b=_cams[METHODS[i]],_cams[METHODS[j]]
        cs=float(a@b/((np.linalg.norm(a)*np.linalg.norm(b))+1e-12))
        print(f'  {METHODS[i]:13s} vs {METHODS[j]:13s}: {cs:.3f}'+('   <-- STILL IDENTICAL' if cs>0.999 else ''))


In [ ]:
# === F3: real 7-method CAM grid -> fig_xai.png (replaces the placeholder Fig 2) ===
import numpy as np, cv2, os
import matplotlib.pyplot as plt
METHODS = ['EigenCAM','GradCAM','GradCAM++','XGradCAM','HiResCAM','LayerCAM','EigenGradCAM']
def _overlay(gray, base_bgr):
    hm = cv2.applyColorMap(np.uint8(255*np.clip(gray,0,1)), cv2.COLORMAP_JET)
    return cv2.addWeighted(base_bgr, 0.55, hm, 0.45, 0)
# pick a confident, correctly-detected image if possible, else the first
_pick = val_images[0]
try:
    for ip in val_images:
        r = model.predict(ip, imgsz=imgsz, conf=0.25, verbose=False, device=DEVICE)[0]
        if r.boxes is not None and len(r.boxes) and float(r.boxes.conf.max())>0.5:
            _pick = ip; break
except Exception as e:
    print('pick fallback:', e)
_img = cv2.imread(_pick); _lb,_,_ = letterbox(_img, imgsz); H,W = _lb.shape[:2]
_cls,_act,_grad = compute_act_grad(model, target_layer, to_tensor(_lb, DEVICE), nc)
fig, axes = plt.subplots(2,4, figsize=(12,6))
axes[0,0].imshow(cv2.cvtColor(_lb, cv2.COLOR_BGR2RGB)); axes[0,0].set_title(f'Input (pred class {_cls})', fontsize=10); axes[0,0].axis('off')
_cells=[axes[0,1],axes[0,2],axes[0,3],axes[1,0],axes[1,1],axes[1,2],axes[1,3]]
for ax,m in zip(_cells,METHODS):
    cam=cv2.resize(cam_from_act_grad(m,_act,_grad),(W,H))
    ax.imshow(cv2.cvtColor(_overlay(cam,_lb),cv2.COLOR_BGR2RGB)); ax.set_title(m,fontsize=10); ax.axis('off')
plt.tight_layout(); plt.savefig('fig_xai.png', dpi=200, bbox_inches='tight')
print(f'saved fig_xai.png from {os.path.basename(_pick)} -> copy it into OMLET_2026/')
